# Session 5: Introduction to physics-informed neural networks

We now have all the ingredients:
- [Sessions 1–2](Session1.ipynb): PDEs and classical numerical solvers.
- [Session 3](Session3.ipynb): neural networks as universal function approximators.
- [Session 4](Session4.ipynb): automatic differentiation — we can compute exact derivatives of neural networks.

This session combines them. We will define the **physics-informed neural network (PINN)** framework precisely, understand its loss function, and implement our first PINN on a 1D ordinary differential equation (ODE) to see it working end-to-end before tackling PDEs in [Session 6](Session6.ipynb).

## 1. The core idea

Consider a general PDE problem:

$$
\mathcal{N}[u](\mathbf{x}, t) = 0, \quad (\mathbf{x}, t) \in \Omega
$$
$$
u(\mathbf{x}, t) = g(\mathbf{x}, t), \quad (\mathbf{x}, t) \in \partial\Omega \cup \{t=0\}
$$

where $\mathcal{N}$ is a differential operator (e.g., $\mathcal{N}[u] = u_t - \alpha u_{xx}$ for the heat equation), $\Omega$ is the domain, and $\partial\Omega$ is its boundary.

**The PINN idea** (Raissi, Perdikaris & Karniadakis, 2019):

> Replace the unknown solution $u(\mathbf{x}, t)$ with a neural network $u_\theta(\mathbf{x}, t)$, then train $\theta$ by minimising the residuals of the PDE, boundary conditions, and initial conditions.

There is no grid, no mesh, and no discretisation of the differential operator. Derivatives are computed exactly via autograd.

## 2. The PINN loss function

The total loss is a sum of three terms:

$$
\mathcal{L}(\theta) = \mathcal{L}_{\text{PDE}} + \mathcal{L}_{\text{BC}} + \mathcal{L}_{\text{IC}}
$$

### PDE residual loss

Sample $N_f$ **collocation points** $\{(\mathbf{x}_i, t_i)\}$ randomly inside the domain. The PDE residual at each point is:

$$
r_i = \mathcal{N}[u_\theta](\mathbf{x}_i, t_i)
$$

$$
\mathcal{L}_{\text{PDE}} = \frac{1}{N_f} \sum_{i=1}^{N_f} r_i^2
$$

### Boundary condition loss

Sample $N_b$ points on $\partial\Omega$:

$$
\mathcal{L}_{\text{BC}} = \frac{1}{N_b} \sum_{j=1}^{N_b} \left( u_\theta(\mathbf{x}_j^{\text{bc}}) - g(\mathbf{x}_j^{\text{bc}}) \right)^2
$$

### Initial condition loss

Sample $N_0$ points at $t=0$:

$$
\mathcal{L}_{\text{IC}} = \frac{1}{N_0} \sum_{k=1}^{N_0} \left( u_\theta(\mathbf{x}_k, 0) - u_0(\mathbf{x}_k) \right)^2
$$

The physics (PDE) acts as a **soft constraint** during training: the network is penalised for violating the equation at every collocation point.

## 3. Collocation points

Collocation points are the locations where we evaluate the PDE residual. They do not have target values — only the equation itself defines the loss. Typical strategies:

- **Uniform random sampling** (Monte Carlo): simplest, works well in low dimensions.
- **Latin Hypercube Sampling**: better space-filling than pure random.
- **Adaptive refinement**: add more points where the residual is large (advanced).

A crucial implementation detail: the collocation points must have `requires_grad=True` so that PyTorch can differentiate the network output with respect to them.

In [17]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Visualise collocation points for a 1D+time problem
torch.manual_seed(0)
N_colloc = 500
x_col = torch.rand(N_colloc)          # x in [0, 1]
t_col = torch.rand(N_colloc)          # t in [0, 1]

N_ic = 50
x_ic = torch.rand(N_ic)
t_ic = torch.zeros(N_ic)

N_bc = 50
x_bc0 = torch.zeros(N_bc)             # left boundary x=0
x_bc1 = torch.ones(N_bc)              # right boundary x=1
t_bc  = torch.rand(N_bc)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_col, t_col, s=4, c='steelblue', label=f'Collocation ({N_colloc})', alpha=0.5)
ax.scatter(x_ic, t_ic, s=15, c='green', label=f'IC points ({N_ic})', zorder=3)
ax.scatter(x_bc0, t_bc, s=15, c='red', label='BC left', zorder=3)
ax.scatter(x_bc1, t_bc, s=15, c='orange', label='BC right', zorder=3)
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('PINN Training Points in (x, t) Space')
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/51/7jyvdh711q54l9wvvmskzjmc0000gn/T/ipykernel_73167/1446370947.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. First PINN: solving a 1D ODE

Let us start with an ODE rather than a PDE, to keep things simple and let us verify against an exact solution.

**Problem**: Solve the exponential decay ODE on $t \in [0, 1]$:
$$
\frac{du}{dt} + k\, u = 0, \quad u(0) = 1
$$
with $k = 5$. The exact solution is $u(t) = e^{-kt}$.

The PINN loss is:
$$
\mathcal{L} = \underbrace{\frac{1}{N_f} \sum_i \left(\frac{du_\theta}{dt}\bigg|_{t_i} + k\, u_\theta(t_i)\right)^2}_{\mathcal{L}_{\text{ODE}}} + \underbrace{\left(u_\theta(0) - 1\right)^2}_{\mathcal{L}_{\text{IC}}}
$$

### 4.1 Network and setup

In [18]:
class PINN_ODE(nn.Module):
    def __init__(self, layers=[1, 32, 32, 1]):
        super().__init__()
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i+1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, t):
        return self.net(t)

k = 5.0
torch.manual_seed(42)

model = PINN_ODE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Collocation points (interior)
N_f = 1000
t_col = torch.rand(N_f, 1, requires_grad=True)   # t in (0, 1)

# Initial condition point
t_ic = torch.zeros(1, 1, requires_grad=False)
u_ic = torch.ones(1, 1)                          # u(0) = 1

### 4.2 Training loop

In [19]:
epochs = 10000
history = []

for epoch in range(epochs):
    optimizer.zero_grad()

    # ----- ODE residual -----
    u_col = model(t_col)                         # u_theta(t)
    du_dt = torch.autograd.grad(
        u_col, t_col,
        grad_outputs=torch.ones_like(u_col),
        create_graph=True
    )[0]
    residual = du_dt + k * u_col                 # du/dt + k*u = 0
    loss_ode = torch.mean(residual**2)

    # ----- Initial condition -----
    loss_ic = torch.mean((model(t_ic) - u_ic)**2)

    loss = loss_ode + loss_ic
    loss.backward()
    optimizer.step()

    history.append(loss.item())
    if epoch % 2000 == 0:
        print(f"Epoch {epoch:6d} | Loss: {loss.item():.2e} | ODE: {loss_ode.item():.2e} | IC: {loss_ic.item():.2e}")

Epoch      0 | Loss: 2.61e+00 | ODE: 1.44e+00 | IC: 1.17e+00
Epoch   2000 | Loss: 1.81e-04 | ODE: 1.80e-04 | IC: 6.84e-07
Epoch   4000 | Loss: 7.90e-05 | ODE: 7.89e-05 | IC: 6.80e-08
Epoch   6000 | Loss: 3.44e-03 | ODE: 3.37e-03 | IC: 6.42e-05
Epoch   8000 | Loss: 2.78e-05 | ODE: 2.78e-05 | IC: 2.98e-09


### 4.3 Results

In [20]:
t_test = torch.linspace(0, 1, 200).reshape(-1, 1)
with torch.no_grad():
    u_pred = model(t_test).numpy()
u_exact = np.exp(-k * t_test.numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_test.numpy(), u_exact, 'k-', linewidth=2, label='Exact: $e^{-5t}$')
axes[0].plot(t_test.numpy(), u_pred, 'r--', linewidth=2, label='PINN')
axes[0].set_xlabel('t')
axes[0].set_ylabel('u(t)')
axes[0].set_title('PINN vs. Exact Solution')
axes[0].legend()

axes[1].semilogy(history)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss')

plt.tight_layout()
plt.show()

l2_error = np.sqrt(np.mean((u_pred - u_exact)**2)) / np.sqrt(np.mean(u_exact**2))
print(f"Relative L2 error: {l2_error:.4e}")

Relative L2 error: 7.0728e-04


/var/folders/51/7jyvdh711q54l9wvvmskzjmc0000gn/T/ipykernel_73167/1930366949.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Animation: standard NN vs. PINN on a damped spring

What happens when data is **sparse and confined to a short window**? The three-panel animation below makes the contrast vivid:

| Panel | What it shows |
|---|---|
| **Left** | Physical spring-mass diagram driven by the exact solution |
| **Centre** | Standard NN prediction evolving **epoch by epoch** (trained on 30 noisy points in $[0, 1.2]$ s only) |
| **Right** | PINN prediction evolving **epoch by epoch** (no measured data — physics constraint enforced on full $[0, 3]$ s) |

**Problem:** Damped harmonic oscillator $\;\ddot{x} + 2\zeta\omega_0\dot{x} + \omega_0^2 x = 0$,
$\;\omega_0 = 2\pi$ rad/s, $\zeta = 0.15$, $x(0)=1$, $\dot{x}(0)=0$.

The animation code lives in `src/spring_animation.py` and is imported below — keeping this notebook clean.

In [ ]:
from src.spring_animation import make_animation
from IPython.display import HTML

ani = make_animation(n_epochs=5000, save_every=50, interval=90)
HTML(ani.to_jshtml())

**What to observe:**

- **Early epochs**: both models produce poor predictions everywhere — neither has learned anything yet.
- **Mid training**: the standard NN rapidly fits the data window (green region), but its prediction beyond $t = 1.2$ s remains wrong or diverges. It has no information about what happens there.
- **Late training**: the PINN converges on the correct decaying oscillation across the *full* domain, driven entirely by the ODE residual loss — no measurements needed in the extrapolation zone.
- The spring panel (left) always displays the true physics, serving as the ground truth.

The core message: **physics constraints act as infinitely dense supervision wherever data is absent.** This is the fundamental advantage of PINNs over purely data-driven approaches.

## 6. Key design choices

| Design choice | Typical setting | Notes |
|---|---|---|
| Activation function | `tanh` | Smooth, supports higher-order autograd |
| Network depth | 3–6 hidden layers | Deeper helps for complex solutions |
| Neurons per layer | 32–128 | Depends on solution complexity |
| Collocation points | $10^3$–$10^5$ | More points → better PDE coverage |
| Optimiser | Adam (then L-BFGS) | Adam for initial training, L-BFGS for fine-tuning |
| Loss weights | $\{\lambda_{\text{PDE}}, \lambda_{\text{BC}}, \lambda_{\text{IC}}\}$ | Often needs tuning; see [Session 8](Session8.ipynb) |

## 7. Summary and what comes next

You have now written your first PINN and seen the key advantage it offers over pure data-driven methods. The structure is always the same:
1. Define the neural network $u_\theta$.
2. Sample collocation points (with `requires_grad=True`).
3. Compute the PDE/ODE residual using autograd.
4. Add boundary/initial condition losses.
5. Optimise with Adam.

**[Session 6](Session6.ipynb)** scales this up to a full PDE: the 1D heat equation. We will deal with two input dimensions $(x, t)$, second-order spatial derivatives, and compare the PINN solution against the analytical result.

**Reading**: Raissi et al. (2019), Sections 1–3. Pay particular attention to their formulation of the physics-informed part of the loss.

## Exercises

1. **Stiffer decay**: repeat the exponential decay experiment with $k = 20$ (ten times stiffer). Does the PINN converge as quickly? Try increasing the number of collocation points and compare convergence. What does this suggest about the difficulty of stiff equations for PINNs?

2. **Simple harmonic oscillator**: extend the ODE PINN to the simple harmonic oscillator $u'' + \omega^2 u = 0$, $u(0) = 1$, $u'(0) = 0$, with $\omega = 2\pi$. The exact solution is $u(t) = \cos(\omega t)$. Note that you will need two initial conditions — one on $u(0)$ and one on $u'(0)$ — so add a second IC loss term for the velocity. Evaluate accuracy over $t \in [0, 2]$.

3. **Remove the IC term**: train the exponential decay PINN without the initial condition loss, keeping only $\mathcal{L}_{\text{ODE}}$. What happens? Why is the IC term essential even though the ODE alone is specified everywhere?

4. **Collocation density**: train the ODE PINN with $N_f \in \{10, 100, 1000, 10000\}$ collocation points. For each, record the final relative $L_2$ error and training time. Plot error vs $N_f$ on a log-log scale. Is there a point of diminishing returns?